# S2 - Fundamentos PySpark: transformaciones, funciones, agrupaciones y evaluación perezosa
**Proyecto Sello · Equipo 10 BIG IMPACT · Predicción de heladas en Juliaca**

Pregunta del proyecto: ¿con qué anticipación y precisión se puede predecir la helada meteorológica (mínima ≤ 0 °C) en Juliaca?

Datos: histórico por hora de WeatherAPI, 2010 – 01-Oct-2026 (146 832 filas).

## 1. Crear la SparkSession

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("sesion2-fundamentos-spark")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/20 01:51:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/20 01:51:43 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


### 1.1 Rutas del proyecto
Se declaran una sola vez. Si cambia la ubicación de los datos, solo se corrige esta celda.

In [2]:
ORIGEN_DATOS = "/opt/u1/s01-arquitectura/data/historico"
ARTIFACTS = "/opt/u1/s02-fundamentos/artifacts"

print("Origen:", ORIGEN_DATOS)
print("Artifacts:", ARTIFACTS)

Origen: /opt/u1/s01-arquitectura/data/historico
Artifacts: /opt/u1/s02-fundamentos/artifacts


## 2. Cargar y explorar el histórico
### 2.1 Lectura con inferSchema
Spark lee los 17 CSV anuales como un solo DataFrame y deduce el tipo de cada columna. Cuesta una pasada extra sobre los datos.

In [3]:
df_clima = spark.read.csv(
    f"{ORIGEN_DATOS}/*.csv",
    header=True,
    inferSchema=True,
)

26/11/20 01:52:43 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /opt/u1/s01-arquitectura/data/historico/*.csv.
java.io.FileNotFoundException: File /opt/u1/s01-arquitectura/data/historico/*.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:210

### 2.2 Primeras filas (show)

In [4]:
df_clima.show(5, truncate=False)

+-------+-------------------+-----------+----------+-------+---------+-------+-------------+
|ciudad |hora               |temperatura|puntoRocio|humedad|nubosidad|presion|precipitacion|
+-------+-------------------+-----------+----------+-------+---------+-------+-------------+
|Juliaca|2016-01-01 00:00:00|4.1        |-0.8      |70     |4        |1017.0 |0.0          |
|Juliaca|2016-01-01 01:00:00|3.7        |-1.5      |69     |3        |1017.0 |0.0          |
|Juliaca|2016-01-01 02:00:00|3.2        |-2.1      |68     |3        |1017.0 |0.0          |
|Juliaca|2016-01-01 03:00:00|2.8        |-2.8      |66     |2        |1018.0 |0.0          |
|Juliaca|2016-01-01 04:00:00|3.1        |-2.4      |60     |2        |1019.0 |0.0          |
+-------+-------------------+-----------+----------+-------+---------+-------+-------------+
only showing top 5 rows


### 2.3 Esquema (printSchema)
Revisar qué tipo asignó Spark a cada columna, sobre todo a `hora`.

In [5]:
df_clima.printSchema()

root
 |-- ciudad: string (nullable = true)
 |-- hora: timestamp (nullable = true)
 |-- temperatura: double (nullable = true)
 |-- puntoRocio: double (nullable = true)
 |-- humedad: integer (nullable = true)
 |-- nubosidad: integer (nullable = true)
 |-- presion: double (nullable = true)
 |-- precipitacion: double (nullable = true)



### 2.4 Resumen estadístico (describe)
Solo de las variables del proyecto, para que la tabla sea legible.

In [6]:
df_clima.select("temperatura", "puntoRocio", "humedad", "nubosidad").describe().show()

26/11/20 01:55:08 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+-----------------+-------------------+------------------+------------------+
|summary|      temperatura|         puntoRocio|           humedad|         nubosidad|
+-------+-----------------+-------------------+------------------+------------------+
|  count|           146832|             146832|            146832|            146832|
|   mean|8.591396289637125|-0.5507825269695975|58.724828375286044|35.298334150593874|
| stddev|4.563461417758553|  5.350886171877291| 22.63550667038643|33.149140696111346|
|    min|             -5.4|              -32.2|                 3|                 0|
|    max|             22.0|                8.9|               100|               100|
+-------+-----------------+-------------------+------------------+------------------+



### 2.5 Una fila en vertical

In [7]:
df_clima.show(2, vertical=True)

-RECORD 0----------------------------
 ciudad        | Juliaca             
 hora          | 2016-01-01 00:00:00 
 temperatura   | 4.1                 
 puntoRocio    | -0.8                
 humedad       | 70                  
 nubosidad     | 4                   
 presion       | 1017.0              
 precipitacion | 0.0                 
-RECORD 1----------------------------
 ciudad        | Juliaca             
 hora          | 2016-01-01 01:00:00 
 temperatura   | 3.7                 
 puntoRocio    | -1.5                
 humedad       | 69                  
 nubosidad     | 3                   
 presion       | 1017.0              
 precipitacion | 0.0                 
only showing top 2 rows


## 3. Lectura con esquema explícito (StructType)
### 3.1 Definir el esquema
Con un StructType el tipo de cada columna lo decido yo y Spark se ahorra la pasada extra de inferencia. `presion` y `precipitacion` se conservan por ahora; la presión queda fuera del modelo por sesgo entre fuentes.

In [8]:
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType, TimestampType
)

schema_clima = StructType([
    StructField("ciudad", StringType(), nullable=True),
    StructField("hora", TimestampType(), nullable=True),
    StructField("temperatura", DoubleType(), nullable=True),
    StructField("puntoRocio", DoubleType(), nullable=True),
    StructField("humedad", IntegerType(), nullable=True),
    StructField("nubosidad", IntegerType(), nullable=True),
    StructField("presion", DoubleType(), nullable=True),
    StructField("precipitacion", DoubleType(), nullable=True),
])

### 3.2 Leer con ese esquema

In [9]:
df_clima2 = spark.read.csv(
    ORIGEN_DATOS,
    header=True,
    schema=schema_clima,
)

### 3.3 Nombres de columna, filas y columnas

In [10]:
print(df_clima2.columns)

num_rows, num_cols = df_clima2.count(), len(df_clima2.columns)
print(f"Filas: {num_rows}, Columnas: {num_cols}")

['ciudad', 'hora', 'temperatura', 'puntoRocio', 'humedad', 'nubosidad', 'presion', 'precipitacion']
Filas: 146832, Columnas: 8


## 4. Muestra aleatoria y escritura de resultados
### 4.1 Muestra del 1 % (sample)
Para explorar y validar la lógica no hace falta trabajar con todo el histórico. Se usa `seed=42` para que la muestra sea reproducible.

In [11]:
df_muestra = df_clima2.sample(withReplacement=False, fraction=0.01, seed=42)
df_muestra.count()

1482

### 4.2 Primeros registros (head)

In [12]:
df_muestra.head(3)

[Row(ciudad='Juliaca', hora=datetime.datetime(2016, 1, 11, 15, 0), temperatura=11.9, puntoRocio=5.6, humedad=66, nubosidad=77, presion=1013.0, precipitacion=0.2),
 Row(ciudad='Juliaca', hora=datetime.datetime(2016, 1, 14, 1, 0), temperatura=5.4, puntoRocio=3.3, humedad=86, nubosidad=27, presion=1019.0, precipitacion=0.0),
 Row(ciudad='Juliaca', hora=datetime.datetime(2016, 1, 17, 11, 0), temperatura=12.1, puntoRocio=6.0, humedad=67, nubosidad=43, presion=1017.0, precipitacion=0.0)]

### 4.3 Guardar la muestra en CSV y Parquet
Spark no escribe un archivo único: escribe una carpeta con un archivo por partición. `overwrite` reemplaza la carpeta si ya existe.

In [13]:
df_muestra.write.mode("overwrite").csv(f"{ARTIFACTS}/clima_muestra_csv", header=True)
df_muestra.write.mode("overwrite").parquet(f"{ARTIFACTS}/clima_muestra_parquet")

### 4.4 Particiones y lectura de vuelta
`getNumPartitions()` indica cuántos archivos `part-` se escribieron. Luego se lee la carpeta completa como un solo DataFrame.

In [14]:
print("Particiones:", df_muestra.rdd.getNumPartitions())

df_leido_csv = spark.read.csv(f"{ARTIFACTS}/clima_muestra_csv", header=True, inferSchema=True)
df_leido_parquet = spark.read.parquet(f"{ARTIFACTS}/clima_muestra_parquet")

print("CSV:", df_leido_csv.count(), "| Parquet:", df_leido_parquet.count())

Particiones: 9
CSV: 1482 | Parquet: 1482


## 5. Transformaciones y evaluación perezosa
### 5.1 select: elegir las variables del proyecto
Una transformación solo construye el plan; no ejecuta nada.

In [15]:
df_variables = df_clima2.select("hora", "temperatura", "puntoRocio", "humedad", "nubosidad")

### 5.2 filter: lecturas con temperatura bajo cero
Aún no se ejecuta nada: `filter` es otra transformación.

In [16]:
from pyspark.sql.functions import col

df_bajo_cero = df_variables.filter(col("temperatura") <= 0)

### 5.3 Encadenar las dos transformaciones
Así se escribe normalmente en código real. Al ejecutar la celda **no aparece ningún dato**, porque todavía no hay una acción.

In [17]:
df_helada_horas = (
    df_clima2
    .select("hora", "temperatura", "puntoRocio", "humedad", "nubosidad")
    .filter(col("temperatura") <= 0)
)

# Hasta aquí Spark solo construyó el plan: no hay salida, no hubo ejecución
df_helada_horas

DataFrame[hora: timestamp, temperatura: double, puntoRocio: double, humedad: int, nubosidad: int]

### 5.4 Acciones: aquí recién Spark ejecuta
`show()` y `count()` son acciones. Cada una dispara un job que se ve en Spark UI (`localhost:4042`).

In [18]:
df_helada_horas.show(10, truncate=False)
df_helada_horas.count()

+-------------------+-----------+----------+-------+---------+
|hora               |temperatura|puntoRocio|humedad|nubosidad|
+-------------------+-----------+----------+-------+---------+
|2016-04-24 03:00:00|-0.1       |-6.7      |61     |4        |
|2016-04-25 02:00:00|-0.2       |-9.3      |50     |0        |
|2016-04-25 03:00:00|-0.7       |-9.5      |51     |0        |
|2016-04-25 04:00:00|-0.4       |-9.6      |46     |0        |
|2016-04-25 06:00:00|-0.2       |-9.9      |37     |0        |
|2016-04-26 01:00:00|-0.2       |-9.4      |50     |0        |
|2016-04-26 02:00:00|-0.5       |-9.3      |51     |0        |
|2016-04-26 03:00:00|-0.8       |-9.2      |53     |0        |
|2016-04-27 05:00:00|-0.4       |-6.8      |50     |1        |
|2016-04-27 06:00:00|-0.5       |-6.8      |45     |2        |
+-------------------+-----------+----------+-------+---------+
only showing top 10 rows


3180

## 6. Plan de ejecución con explain()
### 6.1 explain(True) sobre la consulta encadenada
Se lee de abajo hacia arriba. Hay que comparar el Parsed Logical Plan con el Optimized Logical Plan.

In [19]:
df_helada_horas.explain(True)

== Parsed Logical Plan ==
'Filter '`<=`('temperatura, 0)
+- Project [hora#447, temperatura#448, puntoRocio#449, humedad#450, nubosidad#451]
   +- Relation [ciudad#446,hora#447,temperatura#448,puntoRocio#449,humedad#450,nubosidad#451,presion#452,precipitacion#453] csv

== Analyzed Logical Plan ==
hora: timestamp, temperatura: double, puntoRocio: double, humedad: int, nubosidad: int
Filter (temperatura#448 <= cast(0 as double))
+- Project [hora#447, temperatura#448, puntoRocio#449, humedad#450, nubosidad#451]
   +- Relation [ciudad#446,hora#447,temperatura#448,puntoRocio#449,humedad#450,nubosidad#451,presion#452,precipitacion#453] csv

== Optimized Logical Plan ==
Project [hora#447, temperatura#448, puntoRocio#449, humedad#450, nubosidad#451]
+- Filter (isnotnull(temperatura#448) AND (temperatura#448 <= 0.0))
   +- Relation [ciudad#446,hora#447,temperatura#448,puntoRocio#449,humedad#450,nubosidad#451,presion#452,precipitacion#453] csv

== Physical Plan ==
*(1) Filter (isnotnull(temperatu

### 6.2 Interpretación del plan (se lee de abajo hacia arriba)

**Parsed Logical Plan:** mi código tal cual lo escribí: `Relation` (lee el CSV) → `Project` (mi select) → `Filter` (mi filter). El `Filter` está arriba del `Project`.

**Analyzed Logical Plan:** misma estructura. Spark solo confirmó que las columnas existen y les puso tipo (`temperatura: double`). Además convirtió el `0` en `cast(0 as double)`. Es un chequeo, no una optimización.

**Optimized Logical Plan:** aquí cambia el orden. El `Filter` bajó y quedó pegado a la lectura del archivo; el `Project` pasó arriba. Es el **predicate pushdown**: se descartan filas lo antes posible. También aparecieron dos cambios que yo no escribí:
- `isnotnull(temperatura)`: Catalyst lo agregó porque una comparación `<= 0` nunca es cierta para un nulo.
- `cast(0 as double)` pasó a `0.0`: Spark resolvió la conversión una sola vez en vez de repetirla por fila.

**Physical Plan:** `PushedFilters: [IsNotNull(temperatura), LessThanOrEqual(temperatura,0.0)]` confirma que el filtro se empujó hasta el propio lector del CSV. `ReadSchema` muestra que solo se leen 5 de las 8 columnas (`ciudad`, `presion` y `precipitacion` ni se tocan): eso es **column pruning**.

**Conclusión:** escribí "selecciona y luego filtra"; Spark ejecuta "filtra leyendo solo las columnas necesarias". El resultado es el mismo, procesando menos datos. Es lo que a Expedia le faltó con sus 23.5 TB: filtros que Catalyst pudiera empujar a la lectura.

## 7. Funciones y columnas nuevas con withColumn()
`withColumn()` es una transformación: crea o reemplaza una columna sin ejecutar nada hasta que llegue una acción.

In [21]:
from pyspark.sql.functions import col, when, lit, current_date, year, month

### 7.1 Corregir un tipo con cast()
`temperatura` ya llega como double. Se castea igual para dejar explícito el tipo esperado, sin depender de la inferencia.

In [22]:
df_proc = df_clima2.withColumn("temperatura", col("temperatura").cast("double"))

### 7.2 Clasificar con when()/otherwise(): variable objetivo `hay_helada`
Umbral meteorológico SENAMHI: temperatura ≤ 0 °C. Es la hora con helada; más adelante, en S3/S4, se agrega por día.

In [24]:
df_proc = df_proc.withColumn(
    "hay_helada",
    when(col("temperatura") <= 0, 1).otherwise(0)
)

### 7.3 Clasificar la temperatura en rangos

In [25]:
df_proc = df_proc.withColumn(
    "rango_temp",
    when(col("temperatura") <= 0, "helada")
    .when(col("temperatura") <= 5, "frio")
    .when(col("temperatura") <= 12, "templado")
    .otherwise("calido")
)

### 7.4 Extraer año y mes de la hora (year, month)
Servirán para agrupar por mes en la sección 8.

In [26]:
df_proc = (
    df_proc
    .withColumn("anio", year(col("hora")))
    .withColumn("mes", month(col("hora")))
)

### 7.5 Columna constante y fecha de procesamiento (lit, current_date)

In [27]:
df_proc = (
    df_proc
    .withColumn("fuente", lit("WeatherAPI"))
    .withColumn("fecha_procesado", current_date())
)

### 7.6 Verificar el resultado
Esta acción ejecuta todas las transformaciones anteriores.

In [28]:
df_proc.select(
    "hora", "temperatura", "hay_helada", "rango_temp", "anio", "mes", "fuente", "fecha_procesado"
).show(8, truncate=False)

+-------------------+-----------+----------+----------+----+---+----------+---------------+
|hora               |temperatura|hay_helada|rango_temp|anio|mes|fuente    |fecha_procesado|
+-------------------+-----------+----------+----------+----+---+----------+---------------+
|2016-01-01 00:00:00|4.1        |0         |frio      |2016|1  |WeatherAPI|2026-11-20     |
|2016-01-01 01:00:00|3.7        |0         |frio      |2016|1  |WeatherAPI|2026-11-20     |
|2016-01-01 02:00:00|3.2        |0         |frio      |2016|1  |WeatherAPI|2026-11-20     |
|2016-01-01 03:00:00|2.8        |0         |frio      |2016|1  |WeatherAPI|2026-11-20     |
|2016-01-01 04:00:00|3.1        |0         |frio      |2016|1  |WeatherAPI|2026-11-20     |
|2016-01-01 05:00:00|2.3        |0         |frio      |2016|1  |WeatherAPI|2026-11-20     |
|2016-01-01 06:00:00|3.2        |0         |frio      |2016|1  |WeatherAPI|2026-11-20     |
|2016-01-01 07:00:00|8.3        |0         |templado  |2016|1  |WeatherAPI|2026-

## 8. Agrupaciones y agregaciones
`groupBy()` solo define los grupos; lo que calcula cada grupo es la función de agregación dentro de `.agg()`.

**Advertencia de dominio:** `hay_helada` está a nivel de **hora**. Contar horas con helada no es lo mismo que contar días con helada, que es lo que pregunta el proyecto. Primero se agrupa por mes y después por día.

In [29]:
from pyspark.sql.functions import sum, count, avg, min, max, countDistinct, round

### 8.1 Varias agregaciones a la vez por mes
Un solo `.agg()` con varias funciones hace un solo pase sobre los datos.

In [30]:
df_por_mes = (
    df_proc.groupBy("mes").agg(
        count("*").alias("lecturas"),
        round(avg("temperatura"), 2).alias("temp_media"),
        min("temperatura").alias("temp_min"),
        sum("hay_helada").alias("horas_helada"),
    )
    .orderBy("mes")
)
df_por_mes.show(12)

+---+--------+----------+--------+------------+
|mes|lecturas|temp_media|temp_min|horas_helada|
+---+--------+----------+--------+------------+
|  1|   12648|      9.16|     1.0|           0|
|  2|   11520|      8.94|     0.8|           0|
|  3|   12648|      8.76|    -0.7|           3|
|  4|   12240|      8.12|    -1.7|          77|
|  5|   12648|       7.4|    -4.3|         496|
|  6|   12240|      6.77|    -5.2|        1025|
|  7|   12648|      6.93|    -5.4|         981|
|  8|   12648|      7.92|    -4.3|         466|
|  9|   12240|       9.1|    -4.3|          86|
| 10|   11928|      9.76|    -2.1|          42|
| 11|   11520|     10.63|    -0.5|           4|
| 12|   11904|      9.89|     0.5|           0|
+---+--------+----------+--------+------------+



### 8.2 Mínima diaria (agrupar por día)
La helada meteorológica se define con la temperatura mínima del día. Se agrupa por fecha y se toma el mínimo.

In [31]:
from pyspark.sql.functions import to_date

df_diario = (
    df_proc
    .withColumn("fecha", to_date(col("hora")))
    .groupBy("fecha")
    .agg(min("temperatura").alias("temp_min_dia"))
    .withColumn("dia_helada", when(col("temp_min_dia") <= 0, 1).otherwise(0))
)
df_diario.orderBy("fecha").show(5)

+----------+------------+----------+
|     fecha|temp_min_dia|dia_helada|
+----------+------------+----------+
|2010-01-01|         4.1|         0|
|2010-01-02|         4.5|         0|
|2010-01-03|         5.7|         0|
|2010-01-04|         5.2|         0|
|2010-01-05|         5.6|         0|
+----------+------------+----------+
only showing top 5 rows


### 8.3 Agrupar por varias columnas: porcentaje de días con helada por mes
Se agrupa por mes y se calcula el promedio de `dia_helada`, que equivale a la proporción de días con helada.

In [32]:
from pyspark.sql.functions import month

df_pct_mes = (
    df_diario
    .withColumn("mes", month(col("fecha")))
    .groupBy("mes")
    .agg(
        count("*").alias("dias"),
        sum("dia_helada").alias("dias_helada"),
        round(avg("dia_helada") * 100, 1).alias("pct_dias_helada"),
    )
    .orderBy("mes")
)
df_pct_mes.show(12)

+---+----+-----------+---------------+
|mes|dias|dias_helada|pct_dias_helada|
+---+----+-----------+---------------+
|  1| 527|          0|            0.0|
|  2| 480|          0|            0.0|
|  3| 527|          1|            0.2|
|  4| 510|         37|            7.3|
|  5| 527|        165|           31.3|
|  6| 510|        289|           56.7|
|  7| 527|        289|           54.8|
|  8| 527|        158|           30.0|
|  9| 510|         33|            6.5|
| 10| 497|         15|            3.0|
| 11| 480|          3|            0.6|
| 12| 496|          0|            0.0|
+---+----+-----------+---------------+



### 8.4 agg() con diccionario
Forma alternativa, sin `alias()`. Spark nombra las columnas solo (`sum(...)`, `count(...)`), por eso hace falta `withColumnRenamed()`.

In [33]:
df_agg_dict = (
    df_proc.groupBy("anio")
    .agg({"hay_helada": "sum", "temperatura": "min"})
    .withColumnRenamed("sum(hay_helada)", "horas_helada")
    .withColumnRenamed("min(temperatura)", "temp_min")
    .orderBy("anio")
)
df_agg_dict.show(20)

+----+------------+--------+
|anio|horas_helada|temp_min|
+----+------------+--------+
|2010|         207|    -5.3|
|2011|         229|    -4.4|
|2012|         225|    -3.8|
|2013|         200|    -4.3|
|2014|         160|    -4.3|
|2015|         306|    -5.4|
|2016|         300|    -4.2|
|2017|         145|    -2.5|
|2018|         240|    -2.7|
|2019|         159|    -4.0|
|2020|         127|    -3.0|
|2021|          56|    -2.4|
|2022|         201|    -4.3|
|2023|         249|    -3.2|
|2024|          94|    -3.9|
|2025|          66|    -1.8|
|2026|         216|    -5.2|
+----+------------+--------+



### 8.5 Función ventana (Window): sin colapsar filas
`groupBy().agg()` reduce cada grupo a una sola fila. Una ventana agrega el valor del grupo a cada fila sin perder el detalle. Aquí se agrega la mínima del día a cada lectura horaria.

In [34]:
from pyspark.sql.window import Window

ventana_dia = Window.partitionBy("fecha")

df_con_minima = (
    df_proc
    .withColumn("fecha", to_date(col("hora")))
    .withColumn("temp_min_dia", min("temperatura").over(ventana_dia))
)

df_con_minima.select("hora", "temperatura", "temp_min_dia").orderBy("hora").show(8, truncate=False)

+-------------------+-----------+------------+
|hora               |temperatura|temp_min_dia|
+-------------------+-----------+------------+
|2010-01-01 00:00:00|5.7        |4.1         |
|2010-01-01 01:00:00|4.9        |4.1         |
|2010-01-01 02:00:00|4.7        |4.1         |
|2010-01-01 03:00:00|4.4        |4.1         |
|2010-01-01 04:00:00|4.1        |4.1         |
|2010-01-01 05:00:00|4.9        |4.1         |
|2010-01-01 06:00:00|5.8        |4.1         |
|2010-01-01 07:00:00|7.9        |4.1         |
+-------------------+-----------+------------+
only showing top 8 rows


### 8.6 Comparación: groupBy vs Window
`df_diario` tiene **una fila por día** (6 118 filas). `df_con_minima` conserva **una fila por hora** (146 832 filas), cada una con la mínima de su día repetida. La ventana permite, por ejemplo, calcular cuánto se aleja cada lectura de la mínima del día, cosa que `groupBy` ya no permite porque perdió el detalle.

## 9. RDD: el modelo debajo de los DataFrames
Un DataFrame se apoya en un RDD. En RDD no hay optimización de Catalyst: Spark ejecuta las operaciones tal como se escriben. Se usa `df.rdd` para acceder a él.

### 9.1 Pasar de DataFrame a RDD y descartar nulos

rdd = df_proc.select("hora", "temperatura").rdd.map(lambda r: (r.hora, r.temperatura))
rdd = rdd.filter(lambda x: x[1] is not None)   # descarta lecturas sin temperatura

rdd.take(5)

### 9.2 filter: quedarse solo con las lecturas bajo cero

In [36]:
rdd = df_proc.select("hora", "temperatura").rdd.map(lambda r: (r.hora, r.temperatura))
rdd = rdd.filter(lambda x: x[1] is not None)   # descarta lecturas sin temperatura

rdd.take(5)

[(datetime.datetime(2016, 1, 1, 0, 0), 4.1),
 (datetime.datetime(2016, 1, 1, 1, 0), 3.7),
 (datetime.datetime(2016, 1, 1, 2, 0), 3.2),
 (datetime.datetime(2016, 1, 1, 3, 0), 2.8),
 (datetime.datetime(2016, 1, 1, 4, 0), 3.1)]

### 9.3 map + reduceByKey: horas con helada por mes
`map` convierte cada lectura en un par `(mes, 1)`; `reduceByKey` suma los 1 de cada mes. Es el mismo patrón MapReduce del conteo de palabras. `reduceByKey` provoca un shuffle: mueve pares con la misma clave a la misma partición.

In [38]:
rdd_helada = rdd.filter(lambda x: x[1] <= 0)

rdd_helada.take(5)

[(datetime.datetime(2016, 4, 24, 3, 0), -0.1),
 (datetime.datetime(2016, 4, 25, 2, 0), -0.2),
 (datetime.datetime(2016, 4, 25, 3, 0), -0.7),
 (datetime.datetime(2016, 4, 25, 4, 0), -0.4),
 (datetime.datetime(2016, 4, 25, 6, 0), -0.2)]

### 9.4 Reto: ¿cuántas horas con helada hubo en junio y cuál fue la temperatura mínima registrada?

In [40]:
from operator import add

pares = rdd_helada.map(lambda x: (x[0].month, 1))
conteo_mes = pares.reduceByKey(add)

sorted(conteo_mes.collect())

[(3, 3),
 (4, 77),
 (5, 496),
 (6, 1025),
 (7, 981),
 (8, 466),
 (9, 86),
 (10, 42),
 (11, 4)]

In [41]:
# 1. Horas con helada en junio (mes 6)
print("Junio:", conteo_mes.filter(lambda x: x[0] == 6).collect())

# 2. Las 5 lecturas más frías de todo el histórico
rdd.takeOrdered(5, key=lambda x: x[1])

Junio: [(6, 1025)]


[(datetime.datetime(2015, 7, 8, 6, 0), -5.4),
 (datetime.datetime(2010, 7, 19, 6, 0), -5.3),
 (datetime.datetime(2015, 7, 8, 5, 0), -5.3),
 (datetime.datetime(2015, 7, 8, 4, 0), -5.2),
 (datetime.datetime(2026, 6, 25, 6, 0), -5.2)]

## 10. Error o hallazgo
### 10.1 Horas con helada vs. días con helada
**Qué ocurrió:** al crear `hay_helada` con `when(temperatura <= 0, 1)` y filtrar, obtuve **3 180 lecturas** con helada. Al agrupar por día y tomar la mínima diaria, los días con helada fueron **990**. Mis datos son por hora, así que contar filas con helada no responde la pregunta del proyecto, que es sobre **días**.

**Cómo lo identifiqué:** comparé el conteo del filtro (5.4) con la tabla por día (8.3). La suma de `dias_helada` por mes da 990, muy distinta de 3 180.

**Decisión:** la variable objetivo de U1 se define a nivel de **día**: `dia_helada = 1` si la temperatura mínima del día es ≤ 0 °C (celda 8.2). La columna horaria `hay_helada` se conserva solo para exploración y para la parte de U2 (temperatura de la próxima hora).

**Comportamiento adicional:** al leer con comodín (`*.csv`) Spark mostró un WARN rojo con un stack trace de `FileNotFoundException`. No era un fallo: Spark busca metadatos de streaming y no los encuentra, y después lee los archivos normalmente. Se evita leyendo la carpeta completa.